<!-- nav -->
[← 8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb) · [Home](../../README.md) · [8.17 · SystemVerilog Preprocessor →](08_17_SystemVerilog_Preprocessor.ipynb)

# 8.16 · SystemVerilog Constraints

A SystemVerilog testbench draws random stimulus, and says which values a randomized object may take in constraints
(IEEE 1800 18.5): `x inside {[0:9]}`, `x dist {0 := 1, [1:9] :/ 9}`, `mode == 1 -> len < 8`. Tutorial 8 showed two,
held by `randomize() with` (08.4). This page builds them up in their dialect, `SystemVerilog Constraints`: an expression
held as a constraint, `soft` or not, and a block of them; membership and weighted distributions; implications and
conditionals; a `foreach`, which binds its loop variables; the order, softness and uniqueness constraints. Each is
written in the builders' fluent calls beside the text SystemVerilog writes. Then what is done with them: held by
`randomize() with`, validated, with one refused; never evaluated, since no solver is modeled; and the editions that
have each kind.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), and `refused` gives what an
evaluator says of an expression, its value or the reason it refuses it, as tutorial 8's does.

In [1]:
import assert from "node:assert/strict";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import { Domains, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Constraints = SVConstraints.Expressions.DIALECT; // the dialect, as the framework reads it
const name = (spelling: string) => (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const literal = (value: unknown) => (d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(value)));
const [x, y] = [name("x"), name("y")];


/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.16.1"></a>
## 08.16.1 · An expression held as a constraint

```text
x > 0;        soft x > 0;        { x > 0; y < x; }
```

The simplest constraint is an expression that must be true: an `expressionconstraint`, its `expression` a term of
`Expressions`, and `soft` if a solver may drop it when it conflicts with another (18.5.13). A `constraintblock` holds
constraints in braces, as a class's constraint block does. Each is a kind of `SystemVerilog Constraints`, the root of
its holder, `S.SystemVerilog.Constraints.Expression`, and the family's printer writes each constraint ended by its `;`:

In [2]:
const simple: any[] = store.transaction(() => [
  (c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator(">").right(literal(0n))))),
  (c: any) => c.expressionconstraint((e: any) => e.soft(true).expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator(">").right(literal(0n))))),
  (c: any) => c.constraintblock((b: any) => b.items([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator(">").right(literal(0n))))), (c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(y).operator("<").right(x))))])),
].map((root) => S.SystemVerilog.Constraints.Expression((e: any) => e.root(root))));
for (const constraint of simple) {
  const form = Terms.form(Terms.term(constraint));
  console.log(SystemVerilog.Text.ToText(constraint).padEnd(19), form.kind.padEnd(20), "| its terms:",
    form.arguments.map((t: any) => Terms.kindOf(t)[0]).join(", "), "| free:", [...Symbolics.free(constraint, Constraints)].sort().join(", "));
}
assert.deepEqual(simple.map((c) => SystemVerilog.Text.ToText(c)), ["x > 0;", "soft x > 0;", "{ x > 0; y < x; }"]);

x > 0;              expressionconstraint | its terms: binaryexpression | free: x


soft x > 0;         expressionconstraint | its terms: binaryexpression | free: x


{ x > 0; y < x; }   constraintblock      | its terms: expressionconstraint, expressionconstraint | free: x, y


<a id="T08.16.2"></a>
## 08.16.2 · Membership and distributions

```text
x inside {[0:9]}        x dist {0 := 40, [1:3] :/ 60, default :/ 1}
```

`x inside {[0:9]}` is an expression, `Expressions`' own `insideexpression`, whose set holds a range, `[0:9]`, a
`valuerange`; held as a constraint, it says `x` is among them. A distribution is this dialect's `distexpression`: its
value, an expression, and its items, each a `DistItem` helper holding a value or a range, and an operator and a weight:
`:=` gives each value in it the weight, `:/` divides the weight among them, and an item with no value is `default`
(IEEE 1800-2023):

In [3]:
const inside = (c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.insideexpression((i: any) => i.value(x).set([(d: any) => d.valuerange((r: any) => r.left(literal(0n)).right(literal(9n)))]))));
const membership: any[] = store.transaction(() => [
  inside,
  (c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.distexpression((t: any) => t.value(x).items([
    (i: any) => i.value(literal(0n)).operator(":=").weight(literal(40n)), (i: any) => i.value((d: any) => d.valuerange((r: any) => r.left(literal(1n)).right(literal(3n)))).operator(":/").weight(literal(60n)),
    (i: any) => i.operator(":/").weight(literal(1n))])))),
].map((root) => S.SystemVerilog.Constraints.Expression((e: any) => e.root(root))));
for (const constraint of membership) {
  const expression = Terms.form(Terms.term(constraint)).arguments[0];
  const form = Terms.form(expression);
  console.log(SystemVerilog.Text.ToText(constraint).padEnd(45), form.kind, "of", Constraints.owner(expression).name());
  console.log(" ", [...form.attributes].map(([key, value]) =>
    `${key}: ${(Array.isArray(value) ? value : [value]).map((p: any) => p instanceof Terms.Part ? p.kind : String(p)).join(", ")}`).join(", ") || "-");
}
assert.deepEqual(membership.map((c) => SystemVerilog.Text.ToText(c)), [
  "x inside {[0:9]};", "x dist {0 := 40, [1:3] :/ 60, default :/ 1};"]);

x inside {[0:9]};                             insideexpression of SystemVerilog


  -


x dist {0 := 40, [1:3] :/ 60, default :/ 1};  distexpression of SystemVerilog Constraints


  items: distitem, distitem, distitem


An `inside` belongs to `Expressions`, which this dialect's `Term` lists (tutorial 8, 08.3); a `dist` to this dialect,
and Assertions' `Term` lists it too, since a sequence's expression may be a distribution.

<a id="T08.16.3"></a>
## 08.16.3 · Implications and conditionals

```text
mode == 1 -> len < 8;        if (big) len > 8; else len <= 8;
```

An `implicationconstraint` holds its body where its condition is true; a `conditionalconstraint`, `if`, its consequence
where its condition is true and its alternative, which may be absent, where it is false. Their conditions are
expressions, `Expressions`' terms; their bodies are constraints, this dialect's own, a block among them:

In [4]:
const [mode, len, big] = [name("mode"), name("len"), name("big")];
const implication = (c: any) => c.implicationconstraint((i: any) => i.condition((d: any) => d.binaryexpression((b: any) => b.left(mode).operator("==").right(literal(1n)))).body((c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(len).operator("<").right(literal(8n)))))));
const choices: any[] = store.transaction(() => [
  implication,
  (c: any) => c.conditionalconstraint((i: any) => i.condition(big)
    .consequence((c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(len).operator(">").right(literal(8n)))))).alternative((c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(len).operator("<=").right(literal(8n))))))),
].map((root) => S.SystemVerilog.Constraints.Expression((e: any) => e.root(root))));
for (const constraint of choices) {
  const form = Terms.form(Terms.term(constraint));
  console.log(SystemVerilog.Text.ToText(constraint).padEnd(33), form.kind.padEnd(22),
    form.arguments.map((t: any) => `${Terms.kindOf(t)[0]} of ${Constraints.owner(t).name()}`).join(", "));
}
assert.deepEqual(choices.map((c) => SystemVerilog.Text.ToText(c)), ["mode == 1 -> len < 8;", "if (big) len > 8; else len <= 8;"]);

mode == 1 -> len < 8;             implicationconstraint  binaryexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints


if (big) len > 8; else len <= 8;  conditionalconstraint  nameexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints, expressionconstraint of SystemVerilog Constraints


<a id="T08.16.4"></a>
## 08.16.4 · `foreach` binds its loop variables

```text
foreach (a[i]) a[i] < 10;
```

A `foreach` constrains each element of an array: it binds its loop variables, `i`, the index of each dimension, within
its body alone (18.5.8.1). So it is a binder, a variation of the framework's `let` (tutorial 7): it refines `Let`, its
array Let's `value` and its body Let's `body`, and leaves Let's `name` unset, since it binds a list of variables, which
its kind's scopes give. The body reads `i`, which the `foreach` gives it, and `a`, which the scope must:

In [5]:
const loop = (c: any) => c.foreachconstraint((f: any) => f.value(name("a"))
  .variables([(v: any) => v.identifier((i: any) => i.spelling("i"))])
  .body((c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left((d: any) => d.indexexpression((e: any) => e.value(name("a")).index(name("i")))).operator("<").right(literal(10n)))))));
const each = store.transaction(() => S.SystemVerilog.Constraints.Expression((e: any) => e.root(loop)));
const form = Terms.form(Terms.term(each));
const schema = Terms.kindOf(Terms.term(each))[1].as_schema();
console.log(SystemVerilog.Text.ToText(each), "|", form.kind, "refines", schema.refines.name, "| a binder:", Terms.isBinder(schema) ? "yes" : "no");
console.log("binds", [...SVConstraints.Expressions.loopVariables(Terms.kindOf(Terms.term(each))[1])].sort().join(", "),
  "| value:", SystemVerilog.Text.ToText(form.arguments[0]), "| body:", SystemVerilog.Text.ToText(form.arguments[1]),
  "| free:", [...Symbolics.free(each, Constraints)].sort().join(", "));
assert.equal(SystemVerilog.Text.ToText(each), "foreach (a[i]) a[i] < 10;");
assert.deepEqual([...Symbolics.free(each, Constraints)], ["a"]);

foreach (a[i]) a[i] < 10; | foreachconstraint refines Expressions.Framework.Let | a binder: yes


binds i | value: a | body: a[i] < 10; | free: a


Tutorial 7's list of every binder, read from the store, showed it under `SystemVerilog.Constraints`: the one binder of
SystemVerilog's sublanguages.

<a id="T08.16.5"></a>
## 08.16.5 · Order, softness and uniqueness

```text
solve x before y;        disable soft x;        unique {a, b, c};
```

Three constraints say how the others are solved. `solve ... before` orders the variables a solver picks values for
(18.5.10): it holds two lists of expressions, `solve` and `before`, which its form gives as two parts, each the number of
its terms. `disable soft` drops the soft constraints on its target; `unique` says its expressions, and ranges, have
values different from each other (both IEEE 1800-2012):

In [6]:
const order: any[] = store.transaction(() => [
  (c: any) => c.solvebeforeconstraint((s: any) => s.solve([x]).before([y])),
  (c: any) => c.disablesoftconstraint((d: any) => d.target(x)),
  (c: any) => c.uniqueconstraint((u: any) => u.set([name("a"), name("b"), name("c")])),
].map((root) => S.SystemVerilog.Constraints.Expression((e: any) => e.root(root))));
for (const constraint of order) {
  const form = Terms.form(Terms.term(constraint));
  console.log(SystemVerilog.Text.ToText(constraint).padEnd(18), form.kind.padEnd(22),
    [...form.attributes].map(([key, part]: [string, any]) => `${key} of ${part.size}`).join(", ") || "-", "| terms:", form.arguments.length);
}
assert.deepEqual(order.map((c) => SystemVerilog.Text.ToText(c)), ["solve x before y;", "disable soft x;", "unique {a, b, c};"]);

solve x before y;  solvebeforeconstraint  solve of 1, before of 1 | terms: 2


disable soft x;    disablesoftconstraint  - | terms: 1


unique {a, b, c};  uniqueconstraint       - | terms: 3


<a id="T08.16.6"></a>
## 08.16.6 · Held by `randomize() with`, validated, and one refused

```text
p.randomize() with { x inside {[0:9]}; mode == 1 -> len < 8; foreach (a[i]) a[i] < 10; }
```

A call of `randomize() with` holds its constraints in its `items`, the one property of `Expressions`' kinds that holds
this dialect's `Term`, read from the store. Its host validates the tree, every constraint's names among them, and
`validate` checks each kind's own rules: a distribution's `default` item divides its weight among the values no other
item names, so it weighs them with `:/`, and one with `:=` is refused:

In [7]:
const union = store.registered(Constraints.union);
const host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches as any[];
console.log(host.flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, prop]: [string, any]) => prop.schema === union || prop.schema.element === union)
  .map(([p]: [string, any]) => `${b.name}.${p}`)).join(", "));
const randomized = store.transaction(() => S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.randomizewithexpression((r: any) => r
  .call((c: any) => c.callexpression((m: any) => m.callee((d: any) => d.memberexpression((b: any) => b
    .value(name("p")).member((i: any) => i.spelling("randomize")))).arguments([])))
  .items([inside, implication, loop])))));
const dialect = SystemVerilog.Expressions.DIALECT;
console.log(SystemVerilog.Text.ToText(randomized));
for (const bound of [["p", "x", "mode"], ["p", "x", "mode", "len", "a"]]) {
  console.log(" ", [...bound].sort().join(", "), "|", dialect.validate(randomized, bound).join("; ") || "valid");
}
const fallback = store.transaction(() => S.SystemVerilog.Constraints.Expression((e: any) => e.root((c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.distexpression((t: any) => t.value(x).items([(i: any) => i.operator(":=").weight(literal(1n))])))))));
console.log(SystemVerilog.Text.ToText(fallback), "|", Constraints.validate(fallback, ["x"]).join("; "));
assert.deepEqual(Constraints.validate(fallback, ["x"]), ["expression: a default DistItem weighs its values with :/"]);

randomizewithexpression.items


p.randomize() with { x inside {[0:9]}; mode == 1 -> len < 8; foreach (a[i]) a[i] < 10; }


  mode, p, x | argument 1: body: expression: left: nameexpression 'len' is not bound; argument 2: value: nameexpression 'a' is not bound; argument 2: body: expression: left: value: nameexpression 'a' is not bound


  a, len, mode, p, x | valid


x dist {default := 1}; | expression: a default DistItem weighs its values with :/


<a id="T08.16.7"></a>
## 08.16.7 · Never evaluated

A constraint is solved, not evaluated, and no solver is modeled: SystemVerilog's evaluator refuses each of the
dialect's kinds by name (stage 6, "Q11"), held alone or by `randomize() with`, and a constraint has no domain to infer,
`any`:

In [8]:
const evaluate = SystemVerilog.Simplifiers.OfAny;
for (const constraint of [simple[0], membership[1], choices[0], each, order[0]]) {
  console.log(SystemVerilog.Text.ToText(constraint).padEnd(45), refused((e) => evaluate(e, { x: 1n }), constraint));
}
console.log("inferred:", Constraints.infer(simple[0], { x: Domains.Anything }).name());

x > 0;                                        SystemVerilog Constraints expressionconstraint is not evaluated


x dist {0 := 40, [1:3] :/ 60, default :/ 1};  SystemVerilog Constraints expressionconstraint is not evaluated


mode == 1 -> len < 8;                         SystemVerilog Constraints implicationconstraint is not evaluated


foreach (a[i]) a[i] < 10;                     SystemVerilog Constraints foreachconstraint is not evaluated


solve x before y;                             SystemVerilog Constraints solvebeforeconstraint is not evaluated


inferred: any


<a id="T08.16.8"></a>
## 08.16.8 · Its editions are data

Constraints arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions and none of Verilog's, each with
the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
IEEE 1800-2012 added `disable soft` and `unique`:

In [9]:
const singleton = store.Instances.Expressions.SystemVerilog.Constraints;
const kinds = new Map((store.registered(Constraints.union).branches as any[]).map((branch) => [branch.schema, branch.name]));
for (const [designation, edition] of Standards.standards(singleton)) {
  console.log(designation.padEnd(19), Standards.tags(edition)[1]!.padEnd(19), Standards.kinds(edition).length, "kinds");
}
const [first, second] = ["SystemVerilog-2009", "SystemVerilog-2012"].map((label) => Standards.kinds(Standards.find(singleton, label)!));
console.log("added in IEEE 1800-2012:", second!.filter((k) => !first!.includes(k)).map((k) => kinds.get(k)).join(", "));

IEEE Std 1800-2005  SystemVerilog-2005  7 kinds


IEEE Std 1800-2009  SystemVerilog-2009  7 kinds


IEEE Std 1800-2012  SystemVerilog-2012  9 kinds


IEEE Std 1800-2017  SystemVerilog-2017  9 kinds


IEEE Std 1800-2023  SystemVerilog-2023  9 kinds


added in IEEE 1800-2012: disablesoftconstraint, uniqueconstraint


An edition records the kinds it has, not each of a kind's forms: a distribution's `default` item is IEEE 1800-2023's, but
`distexpression` is in every edition's.

## What you learned

- **A constraint is a term of `SystemVerilog Constraints`**: an expression held as one, `soft` or not, a block, a
  distribution, an implication or a conditional, a `foreach`, and the order, softness and uniqueness constraints; their
  conditions and values are `Expressions`' terms.
- **`foreach` binds its loop variables**, a variation of `let`; nothing else in the dialect binds.
- **`randomize() with` holds them**, and its host validates the tree; a `default` item weighed with `:=` is refused.
- **No solver is modeled**: each kind is refused by name; its editions are IEEE 1800's, two kinds since IEEE 1800-2012.

Next, the directives and macros among them: [SystemVerilog Preprocessor](08_17_SystemVerilog_Preprocessor.ipynb).

<!-- nav -->
[← 8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb) · [Home](../../README.md) · [8.17 · SystemVerilog Preprocessor →](08_17_SystemVerilog_Preprocessor.ipynb)